<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/case-study/fix_duplicate_title.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Fix Duplicate Title - Post

Notebook ini membaca hasil CSV dari `get_posts_h1.ipynb`, lalu melakukan fix langsung ke WordPress via API:

| Kondisi | Aksi |
|---|---|
| H1 pertama di content | **Dihapus** (duplicate dari title field) |
| H1 ke-2, ke-3, dst | **Diubah jadi H2** |

> Upload file CSV hasil dari `get_posts_h1.ipynb` sebelum menjalankan notebook ini.

In [ ]:
import requests
import csv
import json
import time
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
from IPython.display import display, HTML

try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

print("✅ Library berhasil diimport.")

In [ ]:
# ─────────────────────────────────────────────
# 📂 Upload CSV File
# ─────────────────────────────────────────────
print("📂 Silakan upload file CSV hasil dari get_posts_h1.ipynb...\n")

if IN_COLAB:
    uploaded = files.upload()
    csv_filename = list(uploaded.keys())[0]
    print(f"\n✅ File '{csv_filename}' berhasil diupload.")
else:
    # Untuk lokal: ganti dengan path file CSV kamu
    csv_filename = input("Masukkan path file CSV: ").strip()
    print(f"\n✅ Menggunakan file: '{csv_filename}'")

In [ ]:
# ─────────────────────────────────────────────
# 🔑 Load Credentials
# ─────────────────────────────────────────────
wordpress_base_url = "https://www.balitouristic.com/wp-json/wp/v2/posts"

if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("❌ Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)
print("✅ Kredensial berhasil dimuat.")

In [ ]:
# ─────────────────────────────────────────────
# 📖 Baca CSV & Ambil Unique Post IDs
# ─────────────────────────────────────────────
post_ids = []
post_meta = {}  # id -> {url, title, h1_count}

with open(csv_filename, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pid = str(row["ID"]).strip()
        if pid and pid not in post_meta:
            post_ids.append(pid)
            post_meta[pid] = {
                "url": row.get("URL", ""),
                "title": row.get("Title (field)", ""),
                "h1_count": int(row.get("H1 Count", 1))
            }

print(f"📋 Total unique post yang akan diproses: {len(post_ids)}")
print(f"\nContoh 5 pertama:")
for pid in post_ids[:5]:
    m = post_meta[pid]
    print(f"  [{pid}] {m['title'][:60]} | H1 Count: {m['h1_count']}")

In [ ]:
with open(csv_filename, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Headers:", reader.fieldnames)
    # Preview baris pertama
    for row in reader:
        print("Row sample:", dict(row))
        break

In [ ]:
# ─────────────────────────────────────────────
# 🔧 Fungsi: Fix H1 dalam HTML content
# ─────────────────────────────────────────────
def fix_h1_in_content(html_content):
    """
    - H1 pertama  → DIHAPUS (duplicate dari title field)
    - H1 ke-2 dst → diubah jadi H2

    Return: (fixed_html, h1_removed, h1_converted)
    """
    soup = BeautifulSoup(html_content, "html.parser")
    h1_tags = soup.find_all("h1")

    h1_removed   = 0
    h1_converted = 0

    for i, tag in enumerate(h1_tags):
        if i == 0:
            # Hapus H1 pertama
            tag.decompose()
            h1_removed += 1
        else:
            # Ubah H1 berikutnya → H2, pertahankan attributes & inner content
            tag.name = "h2"
            h1_converted += 1

    return str(soup), h1_removed, h1_converted

print("✅ Fungsi fix_h1_in_content siap.")

In [ ]:
# ─────────────────────────────────────────────
# 🚀 Main Loop: Fetch → Fix → Update ke WP
# ─────────────────────────────────────────────
log_success = []
log_failed  = []
log_skipped = []

total = len(post_ids)

print(f"🚀 Memulai proses fix untuk {total} post...\n")
print("=" * 65)

for idx, pid in enumerate(post_ids, start=1):
    meta  = post_meta[pid]
    title = meta['title'][:50]
    print(f"[{idx:>4}/{total}] 🔍 ID {pid} | {title}")

    # 1. Fetch post content dari WP
    try:
        resp = requests.get(
            f"{wordpress_base_url}/{pid}",
            auth=auth,
            params={"_fields": "id,content"}
        )
    except Exception as e:
        print(f"         ❌ Gagal fetch: {e}")
        log_failed.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": str(e)})
        continue

    if resp.status_code != 200:
        print(f"         ❌ HTTP {resp.status_code} saat fetch")
        log_failed.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": f"HTTP {resp.status_code}"})
        continue

    post_data    = resp.json()
    raw_content  = post_data.get("content", {}).get("raw", "") or post_data.get("content", {}).get("rendered", "")

    if not raw_content:
        print(f"         ⬜ Content kosong, skip.")
        log_skipped.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": "Content kosong"})
        continue

    # 2. Fix H1
    fixed_content, removed, converted = fix_h1_in_content(raw_content)

    if removed == 0 and converted == 0:
        print(f"         ⬜ Tidak ada H1 ditemukan, skip.")
        log_skipped.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": "Tidak ada H1"})
        continue

    # 3. Update ke WordPress
    try:
        update_resp = requests.post(
            f"{wordpress_base_url}/{pid}",
            auth=auth,
            json={"content": fixed_content}
        )
    except Exception as e:
        print(f"         ❌ Gagal update: {e}")
        log_failed.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": str(e)})
        continue

    if update_resp.status_code in (200, 201):
        print(f"         ✅ Fixed! H1 dihapus: {removed} | H1→H2: {converted}")
        log_success.append({
            "id": pid,
            "title": meta['title'],
            "url": meta['url'],
            "h1_removed": removed,
            "h1_converted": converted
        })
    else:
        print(f"         ❌ HTTP {update_resp.status_code} saat update")
        log_failed.append({"id": pid, "title": meta['title'], "url": meta['url'], "reason": f"Update HTTP {update_resp.status_code}"})

    # Jeda kecil agar tidak overload server
    time.sleep(0.3)

print("\n" + "=" * 65)

In [ ]:
# ─────────────────────────────────────────────
# 🎉 SUMMARY LOG — Keren Edition
# ─────────────────────────────────────────────
total_h1_removed   = sum(r['h1_removed'] for r in log_success)
total_h1_converted = sum(r['h1_converted'] for r in log_success)

summary_html = f"""
<div style="font-family: 'Segoe UI', sans-serif; max-width: 700px; margin: 16px 0;">

  <div style="background: linear-gradient(135deg, #1a1a2e, #16213e); border-radius: 16px; padding: 28px; color: white; box-shadow: 0 8px 32px rgba(0,0,0,0.3);">
    <div style="font-size: 22px; font-weight: 700; letter-spacing: 1px; margin-bottom: 6px;">🛠️ DUPLICATE TITLE — FIX COMPLETE</div>
    <div style="font-size: 13px; color: #a0aec0; margin-bottom: 24px;">balitouristic.com · WordPress REST API</div>

    <div style="display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 24px;">
      <div style="flex:1; min-width:120px; background: rgba(72,199,142,0.15); border: 1px solid #48c78e; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #48c78e;">{len(log_success)}</div>
        <div style="font-size: 12px; color: #a0aec0; margin-top: 4px;">POST BERHASIL DIFIX</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(255,107,107,0.15); border: 1px solid #ff6b6b; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #ff6b6b;">{total_h1_removed}</div>
        <div style="font-size: 12px; color: #a0aec0; margin-top: 4px;">H1 DIHAPUS</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(99,179,237,0.15); border: 1px solid #63b3ed; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #63b3ed;">{total_h1_converted}</div>
        <div style="font-size: 12px; color: #a0aec0; margin-top: 4px;">H1 → H2</div>
      </div>
      <div style="flex:1; min-width:120px; background: rgba(246,173,85,0.15); border: 1px solid #f6ad55; border-radius: 12px; padding: 16px; text-align:center;">
        <div style="font-size: 36px; font-weight: 800; color: #f6ad55;">{len(log_failed)}</div>
        <div style="font-size: 12px; color: #a0aec0; margin-top: 4px;">GAGAL</div>
      </div>
    </div>

    <div style="background: rgba(255,255,255,0.05); border-radius: 10px; padding: 14px; font-size: 13px; color: #cbd5e0; line-height: 1.8;">
      📋 Total diproses &nbsp;: <b style='color:white'>{total}</b><br>
      ✅ Berhasil difix &nbsp;&nbsp;: <b style='color:#48c78e'>{len(log_success)}</b><br>
      ⬜ Dilewati (skip) : <b style='color:#a0aec0'>{len(log_skipped)}</b><br>
      ❌ Gagal &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;: <b style='color:#ff6b6b'>{len(log_failed)}</b>
    </div>
  </div>

</div>
"""

display(HTML(summary_html))

In [ ]:
# ─────────────────────────────────────────────
# 💾 Download Log CSV (Success + Failed)
# ─────────────────────────────────────────────

# --- Log Success ---
if log_success:
    success_file = "fix_log_success.csv"
    with open(success_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "title", "url", "h1_removed", "h1_converted"])
        writer.writeheader()
        writer.writerows(log_success)
    print(f"💾 Log sukses disimpan: '{success_file}' ({len(log_success)} baris)")
    if IN_COLAB:
        files.download(success_file)

# --- Log Failed ---
if log_failed:
    failed_file = "fix_log_failed.csv"
    with open(failed_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "title", "url", "reason"])
        writer.writeheader()
        writer.writerows(log_failed)
    print(f"💾 Log gagal disimpan : '{failed_file}' ({len(log_failed)} baris)")
    if IN_COLAB:
        files.download(failed_file)

if not log_success and not log_failed:
    print("⚠ Tidak ada data untuk disimpan ke log.")